In [1]:
import sqlite3
import pandas as pd
import json
import sys

# Caminho para seu banco .sqlite
db_path = r"C:\Users\fabio\Documents\GitHub\Easy-Router-Machine\data\processed\streets\streets.sqlite"



# Caminho para o .so ou .dll do SpatiaLite
#mod_spatialite_path = "/usr/lib/mod_spatialite.so"  # Linux
mod_spatialite_path = "mod_spatialite"  # Windows

In [2]:
import os
os.chdir("C:/Users/fabio/Documents/GitHub/Easy-Router-Machine/modules/osmtools/bin/Windows/spatialite/loadable-modules")

In [1]:
import os
os.getcwd()

'c:\\Users\\fabio\\Documents\\GitHub\\Easy-Router-Machine\\dev'

In [3]:
# Conectar no BD
conn = sqlite3.connect(db_path)

In [4]:
#CARREGAR A EXTENSÃO 
conn.enable_load_extension(True)
conn.load_extension(mod_spatialite_path)
cur = conn.cursor()



In [5]:
conn.execute('SELECT load_extension("mod_spatialite.dll")')

In [ ]:
{
  "latitude_origem": "-16.804605",
  "longitude_origem": "-49.206074",
  "latitude_destino": "-16.509934",
  "longitude_destino": "-49.420879"
}

In [ ]:
rota_pref = """
LINESTRING (-49.2055936 -16.802492799999996, -49.2050784 -16.803091200000004, 
-49.2046016 -16.802576, -49.20629439999999 -16.8036064, -49.2057536 -16.806419200000004, 
-49.2141088 -16.8079376, -49.211872 -16.816625599999995, -49.2152896 -16.8262688, -49.2264928 -16.830031999999996, 
-49.241376 -16.823536000000004, -49.2408192 -16.8183264, -49.25345599999999 -16.881961600000004, -49.2545728 -16.9026832, 
-49.2528416 -16.921664, -49.24290559999999 -16.9542576, -49.23099839999999 -16.973100799999997, -49.2281152 -16.994159999999997, 
-49.221318399999994 -17.017134399999996, -49.22480639999999 -17.039686399999997, -49.2256608 -17.0621472, -49.223107199999994 
-17.0857072, -49.217167999999994 -17.1078736, -49.213136 -17.1295808, -49.21536319999999 -17.1522832, -49.2113344 -17.1697136, 
-49.209094400000005 -17.191358399999995, -49.21881919999999 -17.2101376, -49.2296224 -17.2275712, -49.241388799999996 -17.2439568, 
-49.24103999999999 -17.2648976, -49.2356864 -17.287076799999998, -49.238992 -17.334009599999998, -49.24829439999999 -17.353291200000005, 
-49.22366399999999 -17.457603199999994, -49.2153344 -17.473228800000005, -49.2158592 -17.5211024, -49.209820799999996 -17.542734400000004, 
-49.20092799999999 -17.563998399999996, -49.1938848 -17.5877536, -49.18489919999999 -17.6080672, -49.1783328 -17.6257216, -49.1743808 -17.6410976, 
-49.15955199999999 -17.6849152, -49.15511039999999 -17.7057232, -49.1546816 -17.728888, -49.1604512 -17.750865600000004, -49.1716416 -17.7699904, 
-49.188134399999996 -17.7850736, -49.20526399999999 -17.7989424, -49.22461439999999 -17.864780799999995, -49.23302399999999 -17.8875424, -49.2622976 
-18.0013936, -49.264399999999995 -18.0246736, -49.2678336 -18.045694399999995, -49.268355199999995 -18.067695999999994, -49.270595199999995 -18.0883248, 
-49.2772448 -18.109145599999998, -49.28711679999999 -18.155691199999996, -49.2904768 -18.1771808, -49.2822656 -18.1973712, -49.25758079999999 -18.2391696, 
-49.2491712 -18.258987199999996, -49.2067328 -18.3888128, -49.197967999999996 -18.4085792, -49.1938528 -18.425832000000003, -49.1877824 -18.446363199999997, 
-49.188316799999996 -18.468134399999997, -49.18954879999999 -18.480012799999997)"""


In [8]:
query_pref = f"""
WITH
  vars AS (
    SELECT 
      -16.804605 AS lat_o,   -- latitude de origem
      -49.206074 AS long_o,  -- longitude de origem
      -16.509934 AS lat_d,   -- latitude de destino
      -49.420879 AS long_d,  -- longitude de destino
       0.5      AS Box_LatLong  -- filtro de caixa
  ),
  origem AS (
    SELECT node_id AS Node_From
    FROM (
      SELECT node_id,
             ST_Distance(ST_Point(long_o, lat_o), geometry) AS dist
      FROM roads_nodes, vars
      WHERE
            X(geometry) BETWEEN long_o - Box_LatLong AND long_o + Box_LatLong
        AND Y(geometry) BETWEEN lat_o  - Box_LatLong AND lat_o  + Box_LatLong
      ORDER BY dist
      LIMIT 1
    )
  ),
  destino AS (
    SELECT node_id AS Node_To
    FROM (
      SELECT node_id,
             ST_Distance(ST_Point(long_d, lat_d), geometry) AS dist
      FROM roads_nodes, vars
      WHERE
            X(geometry) BETWEEN long_d - Box_LatLong AND long_d + Box_LatLong
        AND Y(geometry) BETWEEN lat_d  - Box_LatLong AND lat_d  + Box_LatLong
      ORDER BY dist
      LIMIT 1
    )
  ),
  -- 1) Defino aqui a sua linestring de referência
  linha AS (
    SELECT
      GeomFromText(
        '{rota_pref}',
        4674
      ) AS geom_line
  ),
  -- 2) Buffer de 300 m em torno da linha
  linha_buf AS (
    SELECT
      ST_Buffer(geom_line, 300) AS geom_buf
    FROM linha
  ),
  -- 3) Calculo peso para cada nó (peso = 300 – distância, se estiver dentro do buffer; senão zero)
  roads_nodes_pref AS (
    SELECT
      rn.node_id,
      CASE
        WHEN ST_Intersects(rn.geometry, lb.geom_buf) THEN
          300 - ST_Distance(rn.geometry, l.geom_line)
        ELSE
          0
      END AS peso
    FROM
      roads_nodes    AS rn
    CROSS JOIN linha      AS l
    CROSS JOIN linha_buf  AS lb
  ),
  -- 4) Ajusto o custo de cada trecho somando os pesos de nodefrom + nodeto
  router_time_pref AS (
    SELECT
      rt.*,
      COALESCE(rnp_f.peso,0) + COALESCE(rnp_t.peso,0) AS peso_total,
      rt.cost + (COALESCE(rnp_f.peso,0) + COALESCE(rnp_t.peso,0)) AS cost_ajustado
    FROM
      router_time   AS rt
    LEFT JOIN roads_nodes_pref AS rnp_f ON rt.NodeFrom = rnp_f.node_id
    LEFT JOIN roads_nodes_pref AS rnp_t ON rt.NodeTo   = rnp_t.node_id
  )
SELECT
  rt.*,
  AsGeoJSON(rt.Geometry) AS geometry_geojson
FROM
  router_time_pref AS rt,
  origem           AS o,
  destino          AS d
WHERE
     rt.NodeFrom = o.Node_From
 AND rt.NodeTo   = d.Node_To
ORDER BY rt.cost_ajustado
LIMIT 1;


"""

In [ ]:

cur.execute(query_pref)
rows = cur.fetchall()

columns = [desc[0] for desc in cur.description]
print(columns)
df = pd.DataFrame(rows, columns=columns)
print(df)

#cur.close()
#conn.close()


In [ ]:


# Consulta SQL com a função AsGeoJSON()
query = """
-- https://www.gaia-gis.it/fossil/libspatialite/wiki?name=VirtualRouting
WITH vars AS (
    SELECT 
        -16.804605 AS lat_o, 	-- latitude de origem
        -49.206074 AS long_o, 	-- longitude de origem
        -16.509934 AS lat_d,  	-- latitude de destino
        -49.420879 AS long_d,   -- longitude de destino
        0.5 AS Box_LatLong      -- Usado para realizar filtro, se diminir aumenta velocidade mas pode não localizar ponto
),
origem AS (
    SELECT node_id as Node_From
    FROM (
        SELECT node_id, 
               ST_Distance(ST_Point(long_o, lat_o), geometry) AS dist
        FROM roads_nodes, vars
        WHERE
                X(geometry) >= long_o   - Box_LatLong AND X(geometry) <= long_o + Box_LatLong
            AND Y(geometry) >= lat_o    - Box_LatLong AND Y(geometry) <= lat_o  + Box_LatLong
        ORDER BY dist ASC
        LIMIT 1
    )
),
destino AS (
    SELECT node_id as Node_To
    FROM (
        SELECT node_id, 
               ST_Distance(ST_Point(long_d, lat_d), geometry) AS dist
        FROM roads_nodes, vars
        WHERE
                X(geometry) >= long_d   - Box_LatLong AND X(geometry) <= long_d + Box_LatLong
            AND Y(geometry) >= lat_d    - Box_LatLong AND Y(geometry) <= lat_d  + Box_LatLong
        ORDER BY dist ASC
        LIMIT 1
    )
)
SELECT *, AsGeoJSON(nc.Geometry) AS geometry_geojson
FROM router_time nc, origem o, destino d
WHERE
	NodeFrom = o.Node_From 
    AND NodeTo = d.Node_To
"""

cur.execute(query)
rows = cur.fetchall()

columns = [desc[0] for desc in cur.description]
print(columns)
df = pd.DataFrame(rows, columns=columns)
print(df)

#cur.close()
#conn.close()


['Algorithm', 'ArcRowid', 'NodeFrom', 'NodeTo', 'Cost', 'Geometry', 'Name', 'Node_From', 'Node_To', 'geometry_geojson']
    Algorithm    ArcRowid  NodeFrom   NodeTo         Cost  \
0    Dijkstra         NaN   2797981  2519534  1820.566371   
1    Dijkstra   9905031.0   2797981  2798372    20.241307   
2    Dijkstra   9905030.0   2798372  2798431     3.226431   
3    Dijkstra   4084534.0   2798431  2791731    43.423343   
4    Dijkstra   4084535.0   2791731  2789868    12.128675   
..        ...         ...       ...      ...          ...   
376  Dijkstra  10969499.0   2523071  2522983     0.821673   
377  Dijkstra  10969500.0   2522983  2522347     5.309002   
378  Dijkstra  10969501.0   2522347  2522251     0.694104   
379  Dijkstra  10969502.0   2522251  2521724     4.127836   
380  Dijkstra  10969503.0   2521724  2519534    15.525145   

                                              Geometry              Name  \
0    b'\x00\x01\xe6\x10\x00\x00@^Y\xb3\xdf\xb5H\xc0...              Non

In [ ]:
import sqlite3
import pandas as pd

# Lista de pontos (latitude, longitude) - ORDEM IMPORTA: origem → waypoint(s) → destino
pontos = [
    (-16.804450, -49.205938),  # origem
    (-16.810834, -49.216445),          # waypoint 1
    (-16.80701, -49.234030)   # destino
]


# Configuração da conexão SQLite com Spatialite
#conn = sqlite3.connect("seu_banco.sqlite")
cur = conn.cursor()

# Tamanho da caixa de busca para encontrar o node mais próximo
box_latlong = 0.5

# Função auxiliar: retorna o node mais próximo de um ponto
def get_nearest_node(lat, lon):
    query = f"""
    SELECT node_id
    FROM (
        SELECT node_id,
               ST_Distance(ST_Point({lon}, {lat}), geometry) AS dist
        FROM roads_nodes
        WHERE X(geometry) BETWEEN {lon - box_latlong} AND {lon + box_latlong}
          AND Y(geometry) BETWEEN {lat - box_latlong} AND {lat + box_latlong}
        ORDER BY dist ASC
        LIMIT 1
    )
    """
    cur.execute(query)
    result = cur.fetchone()
    return result[0] if result else None

# Obter node_id para todos os pontos
nodes = []
for lat, lon in pontos:
    node = get_nearest_node(lat, lon)
    if node is None:
        raise ValueError(f"Nenhum nó encontrado próximo de ({lat}, {lon})")
    nodes.append(node)

# Para cada par consecutivo de nodes, consultar o trajeto
df_total = pd.DataFrame()
for i in range(len(nodes) - 1):
    node_from = nodes[i]
    node_to = nodes[i + 1]

    query = f"""
    SELECT *,
           AsGeoJSON(Geometry) AS geometry_geojson
    FROM router_time
    WHERE NodeFrom = {node_from} AND NodeTo = {node_to}
    """
    cur.execute(query)
    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]
    df_trecho = pd.DataFrame(rows, columns=columns)
    df_total = pd.concat([df_total, df_trecho], ignore_index=True)

# Fechando conexão
cur.close()

# Resultado final com todos os segmentos concatenados
print(df_total[['NodeFrom', 'NodeTo', 'geometry_geojson']])


    NodeFrom   NodeTo                                   geometry_geojson
0    2798184  2785142  {"type":"LineString","coordinates":[[-49.20590...
1    2798184  2798105                                               None
2    2798105  2797981                                               None
3    2797981  2798372                                               None
4    2798372  2798431                                               None
5    2798431  2791731                                               None
6    2791731  2789868                                               None
7    2789868  2789531                                               None
8    2789531  2788628                                               None
9    2788628  2788414                                               None
10   2788414  2788200                                               None
11   2788200  2788073                                               None
12   2788073  2788125                              

In [12]:
df_total

,Algorithm,ArcRowid,NodeFrom,NodeTo,Cost,Geometry,Name,geometry_geojson
0,Dijkstra,NaN,2798184,2785142,204.067110,b'\x00\x01\xe6\x10\x00\x00\x1a\x864\xcf\xb6\x9...,None,"{""type"":""LineString"",""coordinates"":[[-49.20590..."
1,Dijkstra,9905786.0,2798184,2798105,2.961401,None,*** Unknown ****,None
2,Dijkstra,2629370.0,2798105,2797981,1.451065,None,Via Primária 8,None
3,Dijkstra,9905031.0,2797981,2798372,20.241307,None,Via Primária 7,None
4,Dijkstra,9905030.0,2798372,2798431,3.226431,None,Via Primária 7,None
5,Dijkstra,4084534.0,2798431,2791731,43.423343,None,Via Eixo Viário,None
6,Dijkstra,4084535.0,2791731,2789868,12.128675,None,Via Eixo Viário,None
7,Dijkstra,4084536.0,2789868,2789531,2.211979,None,Via Eixo Viário,None
8,Dijkstra,4084537.0,2789531,2788628,5.262885,None,Via Eixo Viário,None
9,Dijkstra,3432953.0,2788628,2788414,1.410510,None,*** Unknown ****,None


In [35]:
geojson_str = df['geometry_geojson'].dropna().iloc[0]
print(geojson_str)

{"type":"LineString","coordinates":[[-49.2716957,-16.7804623],[-49.2738332,-16.7798752],[-49.2751765,-16.7794891],[-49.2762272,-16.7791793],[-49.2763224,-16.7791608],[-49.2763824,-16.7794451],[-49.2764404,-16.7797198],[-49.2765136,-16.7800665],[-49.2765757,-16.7803438],[-49.2766655,-16.7807342],[-49.2767271,-16.780988],[-49.2767442,-16.781053],[-49.276905,-16.7816301],[-49.277049,-16.7821645],[-49.2770833,-16.7822269],[-49.2771223,-16.7822522],[-49.2771653,-16.7822618],[-49.2772635,-16.7822709],[-49.2782115,-16.7820039],[-49.2795567,-16.7816186],[-49.2804713,-16.781372],[-49.2806165,-16.7813921],[-49.2806547,-16.7813902],[-49.280705,-16.7813851],[-49.2809246,-16.7813313],[-49.2810986,-16.7812862],[-49.2811543,-16.7812836],[-49.2811858,-16.7812913],[-49.2812207,-16.7813033],[-49.2812463,-16.7813341],[-49.2812622,-16.78136],[-49.2812698,-16.7813942],[-49.2812663,-16.7814281],[-49.281249,-16.7814781],[-49.2812133,-16.7815276],[-49.2811744,-16.7815552],[-49.2811006,-16.7815841],[-49.280985

In [ ]:
geojson_obj = json.loads(geojson_str)

In [67]:
geojson_binario = df['Geometry'].dropna().iloc[0]


In [6]:
import random

def gerar_coords_goias():
    lat = random.uniform(-18.0, -12.0)
    lon = random.uniform(-51.0, -46.0)
    return lat, lon

In [8]:
def montar_query(lat_o, lon_o, lat_d, lon_d):
    return f"""
    WITH vars AS (
        SELECT 
            {lat_o} AS lat_o,
            {lon_o} AS long_o,
            {lat_d} AS lat_d,
            {lon_d} AS long_d,
            0.5 AS Box_LatLong
    ),
    origem AS (
        SELECT node_id as Node_From
        FROM (
            SELECT node_id,
                   ST_Distance(ST_Point(long_o, lat_o), geometry) AS dist
            FROM roads_nodes, vars
            WHERE
                    X(geometry) >= long_o - Box_LatLong AND X(geometry) <= long_o + Box_LatLong
                AND Y(geometry) >= lat_o - Box_LatLong AND Y(geometry) <= lat_o + Box_LatLong
            ORDER BY dist ASC
            LIMIT 1
        )
    ),
    destino AS (
        SELECT node_id as Node_To
        FROM (
            SELECT node_id,
                   ST_Distance(ST_Point(long_d, lat_d), geometry) AS dist
            FROM roads_nodes, vars
            WHERE
                    X(geometry) >= long_d - Box_LatLong AND X(geometry) <= long_d + Box_LatLong
                AND Y(geometry) >= lat_d - Box_LatLong AND Y(geometry) <= lat_d + Box_LatLong
            ORDER BY dist ASC
            LIMIT 1
        )
    )
    SELECT *, AsGeoJSON(nc.Geometry) AS geometry_geojson
    FROM router_time nc, origem o, destino d
    WHERE NodeFrom = o.Node_From AND NodeTo = d.Node_To
    """

In [2]:
import sqlite3
import pandas as pd
import json

# Caminho para seu banco .sqlite
db_path = r"C:\Users\fabio\Documents\GitHub\Easy-Router-Machine\data\processed\streets\streets.sqlite"

# Caminho para o .so ou .dll do SpatiaLite
#mod_spatialite_path = "/usr/lib/mod_spatialite.so"  # Linux
mod_spatialite_path = "mod_spatialite.dll"  # Windows

# Conectar no BD
conn = sqlite3.connect(db_path)

#CARREGAR A EXTENSÃO 
conn.enable_load_extension(True)
conn.load_extension(mod_spatialite_path)

cur = conn.cursor()

conn.execute('SELECT load_extension("mod_spatialite.dll")')

In [3]:
import sqlite3



def executar_rota(i):
    lat_o, lon_o = gerar_coords_goias()
    lat_d, lon_d = gerar_coords_goias()
    query = montar_query(lat_o, lon_o, lat_d, lon_d)
    try:
        db_path = r"C:\Users\fabio\Documents\GitHub\Easy-Router-Machine\data\processed\streets\streets.sqlite"
        mod_spatialite_path = "mod_spatialite.dll"  # Windows
        # Cada thread cria sua própria conexão
        conn = sqlite3.connect(db_path, check_same_thread=False)
        conn.enable_load_extension(True)
        conn.load_extension(mod_spatialite_path)
        conn.execute('SELECT load_extension("mod_spatialite.dll")')
        cur = conn.cursor()
        cur.execute(query)
        rows = cur.fetchall()
        conn.close()
        columns = [desc[0] for desc in cur.description]
        df = pd.DataFrame(rows, columns=columns)
        print(f"[{i}] Rota executada com sucesso.")
        geojson = df['geometry_geojson'].dropna().iloc[0]
        return  geojson
    except Exception as e:
        print(f"[{i}] ERRO: {e}")

In [117]:
ROTA = executar_rota(1)
print(ROTA)

[1] Rota executada com sucesso.
{"type":"LineString","coordinates":[[-47.7711645,-15.4188055],[-47.7712446,-15.419352],[-47.7713309,-15.419851],[-47.7714574,-15.4201317],[-47.7712874,-15.4205457],[-47.7708666,-15.4212626],[-47.7706736,-15.4216715],[-47.77055,-15.422444],[-47.7706019,-15.4225234],[-47.7707636,-15.4228713],[-47.7709845,-15.4232482],[-47.7718483,-15.4238541],[-47.7721112,-15.4243609],[-47.7725485,-15.4247664],[-47.7734649,-15.4249974],[-47.7737528,-15.4251683],[-47.7741158,-15.4252737],[-47.7750072,-15.4253016],[-47.7755215,-15.4259052],[-47.7761011,-15.4263831],[-47.7760789,-15.4268275],[-47.7763867,-15.4275651],[-47.7768017,-15.4283384],[-47.7770573,-15.4285424],[-47.7776235,-15.4287491],[-47.7780819,-15.4296103],[-47.7784091,-15.4299128],[-47.7785593,-15.4302087],[-47.7788302,-15.4302592],[-47.7793611,-15.4301046],[-47.7796751,-15.4302179],[-47.7802386,-15.4307699],[-47.7807051,-15.4308519],[-47.7810243,-15.4311487],[-47.7814852,-15.4314224],[-47.7820824,-15.4313498],[

In [15]:
import concurrent.futures
import time
import json

def teste_sobrecarga(num_threads=50, output_file='resultados.json'):
    inicio = time.time()
    resultados = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(executar_rota, i) for i in range(num_threads)]
        for future in concurrent.futures.as_completed(futures):
            try:
                resultado = future.result()
                print(resultado)
                resultados.append(resultado)

            except Exception as e:
                print(f"Erro na thread: {e}")
    fim = time.time()
    print(f"Teste com {num_threads} threads finalizado em {fim - inicio:.2f} segundos.")

    # Salva os resultados no arquivo JSON
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Carregar os pedágios do banco 
df = pd.DataFrame(columns=['NOME_PEDAGIO', 'latitude', 'longitude'])

# Adicionando linhas (dados)
df.loc[0] = ['PEDAGIO 153 1', '-16.799962', '-49.238545']
df.loc[1] = ['av 1', '-16.809789', '-49.222553'] 


# 2. Converter para GeoDataFrame
gdf_pedagios = gpd.GeoDataFrame(
    df,
    geometry=[Point(lon, lat) for lon, lat in zip(df['longitude'], df['latitude'])],
    crs="EPSG:4326"  # Sistema geográfico padrão (WGS 84)
)

In [26]:
import requests
from requests.auth import HTTPBasicAuth
import json

def gerar_rota(origem_lat, origem_long, destino_lat, destino_long):
    url = "https://apipex.dev.br/router/v0/routes/"

    headers = {
        "accept": "application/json",
        "Content-Type": "application/json"
    }

    data = {
        "latitude_origem": f'{origem_lat}',
        "longitude_origem": f'{origem_long}',
        "latitude_destino": f'{destino_lat}',
        "longitude_destino": f'{destino_long}'
    }

    usuario = "equipe_cca"
    senha = "Cca@2024"
    response = requests.post(url, json=data, headers=headers, auth=HTTPBasicAuth(usuario, senha))
    response_text = response.text
    geojson_dict = json.loads(response_text)["DADOS"]
    
    return geojson_dict


In [5]:
# 3. Carregar a rota (GeoJSON com LineString)
geojson_dict = gerar_rota("-16.804450", "-49.205938", "-16.792611", "-49.237988")

In [130]:
import json
from shapely.geometry import shape

# 4. Converte para objeto shapely
geometry = shape(geojson_dict)

# 5. Cria GeoDataFrame
gdf_rota = gpd.GeoDataFrame(geometry=[geometry], crs="EPSG:4326")

# 6. Converte para sistema métrico as rotas (Web Mercator)
gdf_rota = gdf_rota.to_crs(epsg=3857)

In [ ]:
# 7. Converte para sistema métrico os pedagios (Web Mercator)
gdf_pedagios = gdf_pedagios.to_crs(epsg=3857)


In [132]:
# 8. Criar buffer da rota (ex: 1000 metros de raio)
buffer_rota = gdf_rota.geometry.buffer(200).unary_union

C:\Users\fabio\AppData\Local\Temp\ipykernel_10064\109403456.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  buffer_rota = gdf_rota.geometry.buffer(200).unary_union


In [133]:
# 9. Verificar se pedágios estão dentro do buffer
gdf_pedagios['proximo'] = gdf_pedagios.geometry.within(buffer_rota)

In [136]:
gdf_pedagios

,NOME_PEDAGIO,latitude,longitude,geometry,proximo
0,PEDAGIO 153 1,-16.799962,-49.238545,POINT (-5481209.757 -1897551.814),True
1,av 1,-16.809789,-49.222553,POINT (-5479429.536 -1898694.551),True


In [137]:
# 10. Filtrar apenas os próximos
pedagios_proximos = gdf_pedagios[gdf_pedagios['proximo'] == True]

In [3]:
################ Teste pelo sqlite

import os
os.chdir("C:/Users/fabio/Documents/GitHub/Easy-Router-Machine/modules/osmtools/bin/Windows/spatialite/loadable-modules")



In [ ]:
import sqlite3
import pandas as pd
import os

# 1. Criar DataFrame com os pedágios
df = pd.DataFrame(columns=['NOME_PEDAGIO', 'latitude', 'longitude'])
df.loc[0] = ['PEDAGIO 153 1', '-16.799962', '-49.238545']


# 2. Caminho do banco de dados
db_path = 'pedagios_spatialite.db'

# 3. Remover banco anterior (opcional)
if os.path.exists(db_path):
    os.remove(db_path)
    
path_proj= r'C:\Users\fabio\Documents\GitHub\Easy-Router-Machine\modules\osmtools\bin\Windows\spatialite\loadable-modules\proj.db'

# Montar e executar a query
query_proj = f"SELECT PROJ_SetDatabasePath('{path_proj}');"

# 4. Conectar ao SQLite e carregar extensão Spatialite
conn = sqlite3.connect(db_path)
conn.enable_load_extension(True)

try:
    conn.load_extension('mod_spatialite.dll')
except Exception as e:
    raise RuntimeError("Erro ao carregar o mod_spatialite: ", e)

cur = conn.cursor()
cur.execute(query_proj)

# 5. Criar tabela e coluna geométrica
cur.executescript('''
    SELECT InitSpatialMetadata();

    CREATE TABLE infraestrutura_rodoviaria (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                tipo TEXT,
                nome TEXT,
                data_cadastro TEXT
                 
    );

    SELECT AddGeometryColumn('infraestrutura_rodoviaria', 'geom', 4326, 'POINT', 'XY');
''')

# 6. Inserir os dados do DataFrame
for _, row in df.iterrows():
    nome = row['NOME_PEDAGIO']
    lat = float(row['latitude'])
    lon = float(row['longitude'])
    wkt_point = f"POINT({lon} {lat})"
    
    cur.execute('''
        INSERT INTO pedagios (nome, geom)
        VALUES (?, ST_GeomFromText(?, 4326))
    ''', (nome, wkt_point))

conn.commit()

# 7. Verificar inserções
for row in cur.execute("SELECT id, nome, AsText(geom) FROM pedagios;"):
    print(row)

#conn.close()

(1, 'PEDAGIO 153 1', 'POINT(-49.238545 -16.799962)')
(2, 'av 1', 'POINT(-49.222553 -16.809789)')


In [2]:
for row in cur.execute("SELECT id, nome, AsText(geom) FROM pedagios;"):
    print(row)

OperationalError: no such table: pedagios

In [43]:
# 3. Carregar a rota (GeoJSON com LineString)
geojson_dict = gerar_rota("-16.804450", "-49.205938", "-16.616516", "-49.205985")
# AV GOIAS -16.826507, -49.288806

# AEROPORTO -16.616516, -49.205985

In [48]:
print(geojson_dict)

{'type': 'LineString', 'coordinates': [[-49.2059022, -16.8043511], [-49.205929, -16.8043844], [-49.2059512, -16.8044483], [-49.2059657, -16.80456], [-49.2060774, -16.8045798], [-49.2057392, -16.8060618], [-49.2056871, -16.8062984], [-49.2101791, -16.8072011], [-49.2112304, -16.8074202], [-49.2122465, -16.8076319], [-49.2127714, -16.8077637], [-49.2130532, -16.8078234], [-49.2135579, -16.8079178], [-49.2137286, -16.8079413], [-49.2137914, -16.8078769], [-49.2138757, -16.8078416], [-49.2139363, -16.8078371], [-49.2140443, -16.8078701], [-49.214108, -16.8079272], [-49.2141446, -16.808003], [-49.2142712, -16.8080867], [-49.2143963, -16.8081375], [-49.2152428, -16.8083091], [-49.2211217, -16.8095289], [-49.2231706, -16.809954], [-49.2233868, -16.8099718], [-49.2235845, -16.809956], [-49.2236712, -16.8098963], [-49.223777, -16.809883], [-49.2238768, -16.8099192], [-49.2239469, -16.8099963], [-49.2240282, -16.810022], [-49.2241819, -16.8100539], [-49.2243246, -16.8100656], [-49.2248129, -16.8

In [5]:
from shapely.geometry import shape
from shapely.wkt import dumps as to_wkt


# 2. Converte GeoJSON -> WKT
rota_geom = shape(geojson_dict)
wkt_line = to_wkt(rota_geom)

NameError: name 'geojson_dict' is not defined

In [4]:
wkt_line = 'LINESTRING (-49.2059021999999970 -16.8043511000000017, -49.2059289999999976 -16.8043844000000000, -49.2059512000000012 -16.8044483000000007, -49.2059657000000001 -16.8045599999999986, -49.2060773999999981 -16.8045797999999991, -49.2057391999999965 -16.8060617999999984, -49.2056870999999987 -16.8062983999999993, -49.2101790999999977 -16.8072011000000003, -49.2112303999999980 -16.8074201999999993, -49.2122464999999991 -16.8076319000000005, -49.2127714000000012 -16.8077636999999989, -49.2130531999999974 -16.8078234000000002, -49.2135578999999979 -16.8079177999999985, -49.2137286000000032 -16.8079412999999995, -49.2137913999999981 -16.8078769000000001, -49.2138757000000027 -16.8078415999999997, -49.2139363000000003 -16.8078371000000004, -49.2140442999999976 -16.8078700999999988, -49.2141080000000031 -16.8079272000000017, -49.2141445999999974 -16.8080029999999994, -49.2142711999999989 -16.8080867000000005, -49.2143962999999971 -16.8081375000000008, -49.2152427999999986 -16.8083090999999989, -49.2211216999999976 -16.8095289000000001, -49.2231706000000031 -16.8099540000000012, -49.2233868000000001 -16.8099717999999996, -49.2235845000000012 -16.8099559999999997, -49.2236711999999983 -16.8098962999999983, -49.2237769999999983 -16.8098829999999992, -49.2238767999999993 -16.8099191999999995, -49.2239469000000014 -16.8099963000000017, -49.2240281999999993 -16.8100220000000000, -49.2241818999999978 -16.8100538999999998, -49.2243246000000028 -16.8100656000000015, -49.2248129000000034 -16.8100717999999993, -49.2253816999999998 -16.8100464000000009, -49.2259144000000006 -16.8099738000000016, -49.2264168000000026 -16.8098793999999998, -49.2268888999999987 -16.8097542000000004, -49.2313508000000013 -16.8083248999999988, -49.2317828999999989 -16.8081846000000006, -49.2319973000000033 -16.8081117000000013, -49.2326647000000008 -16.8078359000000006, -49.2336863000000022 -16.8073048999999983, -49.2339155000000019 -16.8071850000000005, -49.2339893999999987 -16.8071215999999986, -49.2340261000000012 -16.8070662999999989, -49.2340185999999989 -16.8069932000000009, -49.2340360999999973 -16.8069215000000014, -49.2340766000000016 -16.8068591000000005, -49.2341357999999971 -16.8068123000000007, -49.2342076999999989 -16.8067862000000012, -49.2342843999999999 -16.8067835000000017, -49.2343580000000003 -16.8068044999999984, -49.2344207000000011 -16.8068470000000012, -49.2344658999999965 -16.8069065000000002, -49.2346821000000006 -16.8068598999999992, -49.2349570000000014 -16.8067329000000001, -49.2365441000000033 -16.8057705000000013, -49.2366093999999990 -16.8057316000000014, -49.2374632000000005 -16.8052239999999991, -49.2386612000000028 -16.8045116999999991, -49.2389563999999993 -16.8043362000000016, -49.2389189000000016 -16.8041340999999989, -49.2388826999999978 -16.8039145999999988, -49.2387968000000029 -16.8034703999999984, -49.2387378000000027 -16.8030377000000009, -49.2386399000000026 -16.8025434000000011, -49.2385930000000016 -16.8022533000000003, -49.2385232000000030 -16.8017949000000009, -49.2384629000000018 -16.8012633999999998, -49.2384565999999992 -16.8011118999999987, -49.2384428000000014 -16.8007780999999987, -49.2384346999999991 -16.8003788000000007, -49.2384361000000013 -16.8000553000000004, -49.2384508000000025 -16.7997164000000012, -49.2384508000000025 -16.7992144000000003, -49.2384374000000022 -16.7987830000000002, -49.2383382000000012 -16.7976840000000003, -49.2382455999999991 -16.7966030000000011, -49.2382094000000023 -16.7962743000000003, -49.2382930999999999 -16.7957564999999995, -49.2382537000000013 -16.7952574000000006, -49.2382088999999965 -16.7947917000000011, -49.2381578999999974 -16.7942031999999983, -49.2380944000000014 -16.7933705000000018, -49.2380630999999980 -16.7929971999999985)'

In [6]:
import json

query = f'''
SELECT 
    p.id, p.nome, 
    AsGeoJSON(p.geom)
FROM 
    infraestrutura_rodoviaria p
WHERE 
    ST_Within( -- funcao de comparação de geometrias
        ST_Transform(p.geom, 3857),  -- primeira geometria
        ST_Buffer(ST_Transform(ST_GeomFromText('{wkt_line}', 4326), 3857 ), 200 ) -- segunda geometria(buffer)
);

'''

cur.execute(query)
results = cur.fetchall()


df = pd.DataFrame(results, columns=['id', 'nome', 'geom_geojson'])

print(df)

OperationalError: ST_Transform exception - PROJ reports "proj_create: no database context specified".

In [ ]:


# 7. Exibir resultados como GeoJSON Features
# features = []
# for row in results:
#     features.append({
#         "type": "Feature",
#         "geometry": json.loads(row[2]),
#         "properties": {
#             "id": row[0],
#             "nome": row[1]
#         }
#     })

# geojson_result = {
#     "type": "FeatureCollection",
#     "features": features
# }

# 8. Exibir
# print(json.dumps(geojson_result, indent=2, ensure_ascii=False))

#cur.close()
#conn.close()

,id,nome,geom_geojson
0,1,PEDAGIO 153 1,"{""type"":""Point"",""coordinates"":[-49.238545,-16...."
1,2,av 1,"{""type"":""Point"",""coordinates"":[-49.222553,-16...."


In [19]:
query_pedagio_furnas = f'''
INSERT INTO pedagios (nome, geom)
VALUES ('Pedágio Furnas', ST_GeomFromText('POINT(-49.243445 -16.812842)', 4326));
'''
cur.execute(query_pedagio_furnas)
conn.commit()

In [21]:
query_pedagio_aeroporto = f'''
INSERT INTO pedagios (nome, geom)
VALUES ('Pedágio Aeroporto de goiania', ST_GeomFromText('POINT(-49.205871 -16.638437)', 4326));
'''
cur.execute(query_pedagio_aeroporto)
conn.commit()

In [2]:
import pygeoif

In [4]:
import geojson
from shapely.geometry import shape

In [5]:
import sys
print(sys.executable)

c:\Users\fabio\anaconda3\envs\Brabo\python.exe


In [1]:
# TESTE BANCO DE DADOS LOCALES
import os
import sqlite3
import pandas as pd

os.chdir("C:/Users/fabio/Documents/GitHub/Easy-Router-Machine/modules/osmtools/bin/Windows/spatialite/loadable-modules")

db_path = r'C:\Users\fabio\Documents\GitHub\Easy-Router-Machine\data\processed\streets\db_locales.sqlite'
conn = sqlite3.connect(db_path)
conn.enable_load_extension(True)
try:
    conn.load_extension('mod_spatialite.dll')
except Exception as e:
    raise RuntimeError("Erro ao carregar o mod_spatialite: ", e)


# 1. Criar DataFrame com os pedágios
df = pd.DataFrame(columns=['NOME_PEDAGIO', 'latitude', 'longitude'])
df.loc[0] = ['PEDAGIO 153 1', '-16.799962', '-49.238545']
df.loc[1] = ['av 1', '-16.809789', '-49.222553']


cur = conn.cursor()

# 6. Inserir os dados do DataFrame
for _, row in df.iterrows():
    nome = row['NOME_PEDAGIO']
    lat = float(row['latitude'])
    lon = float(row['longitude'])
    wkt_point = f"POINT({lon} {lat})"
    
    cur.execute('''
        INSERT INTO infraestrutura_rodoviaria (nome, geom)
        VALUES (?, ST_GeomFromText(?, 4326))
    ''', (nome, wkt_point))

conn.commit()

# 7. Verificar inserções
for row in cur.execute("SELECT id, nome, AsText(geom) FROM infraestrutura_rodoviaria;"):
    print(row)



(1, 'PEDAGIO 153 1', 'POINT(-49.238545 -16.799962)')
(2, 'av 1', 'POINT(-49.222553 -16.809789)')


In [3]:
# 7. Verificar inserções
for row in cur.execute("SELECT id, nome, AsText(geom) FROM infraestrutura_rodoviaria;"):
    print(row)

(1, 'PEDAGIO 153 1', 'POINT(-49.238545 -16.799962)')
(2, 'av 1', 'POINT(-49.222553 -16.809789)')


In [ ]:
!where python




c:\Users\fabio\anaconda3\envs\Brabo\python.exe
C:\Users\fabio\anaconda3\python.exe
C:\Users\fabio\AppData\Local\Microsoft\WindowsApps\python.exe


In [11]:
import json

query = f'''
SELECT 
    p.id, p.nome, 
    AsGeoJSON(p.geom)
FROM 
    infraestrutura_rodoviaria p
WHERE 
    ST_Within(
        p.geom,
        ST_Buffer(ST_GeomFromText('{wkt_line}', 4326), 0.0018) -- buffer ~200m em graus (aprox)
    );


'''

cur.execute(query)
results = cur.fetchall()


df = pd.DataFrame(results, columns=['id', 'nome', 'geom_geojson'])

print(df)

Empty DataFrame
Columns: [id, nome, geom_geojson]
Index: []
